# recs_004 task A (consumer mode) - _003

This notebook is analysis-only and consumes centralized retrieval eval artifacts.

## Interpretation notes (from original recs_004 learnings)

- Popularity deciles: `multi_mean_train` tends to lead in `D1-D8` (tail-to-mid popularity), while `popularity_train` tends to win in `D9-D10` (head), which is expected.
- Slice behavior:
  - `slice_a_multi_target` (`n_eval_targets >= 2`): `multi_mean_train` generally beats `popularity_train`.
  - `slice_b_single_target` (`n_eval_targets == 1`): `popularity_train` generally leads.
- Support intuition: as train-history signal grows (more/cleaner support behavior and review text), `multi_mean_train` should improve relative to popularity-heavy baselines.

Use these as priors while reading the tables below; verify against current run outputs and avoid over-generalizing from any single run.

## Prerequisites

Run pipeline first:
- `python scripts/recs_job_eval_retrieval.py configs/recs_job_eval_retrieval.json`

Optional baseline freeze:
- `python scripts/recs_job_eval_retrieval.py configs/recs_job_eval_retrieval.json --write-baseline`

Artifact source:
- `artifacts/recs/retrieval/runs/latest/`

In [1]:
from pathlib import Path
import json

import pandas as pd
# no limit on columns shown in pandas
pd.set_option("display.max_columns", None)

def _find_repo_root(start: Path) -> Path:
    here = start.resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root from start={start}")


REPO_ROOT = _find_repo_root(Path.cwd())
EVAL_DIR = REPO_ROOT / "artifacts" / "recs" / "retrieval" / "runs" / "latest"

PATHS = {
    "overall": EVAL_DIR / "eval_retrieval_overall.csv",
    "by_slice": EVAL_DIR / "eval_retrieval_by_slice.csv",
    "by_support": EVAL_DIR / "eval_retrieval_by_support_bucket.csv",
    "by_pop_decile": EVAL_DIR / "eval_retrieval_by_pop_decile.csv",
    "pop_delta": EVAL_DIR / "eval_retrieval_pop_delta_vs_popularity.csv",
    "personalization": EVAL_DIR / "eval_retrieval_personalization.csv",
    "run_meta": EVAL_DIR / "eval_retrieval_run_meta.json",
}

missing = [str(p) for p in PATHS.values() if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing eval artifact(s):\n- " + "\n- ".join(missing))

tables = {k: pd.read_csv(v) for k, v in PATHS.items() if k != "run_meta"}
run_meta = json.loads(PATHS["run_meta"].read_text(encoding="utf-8"))

print("Loaded artifacts from", EVAL_DIR)
print("Methods:", run_meta.get("methods_run", []))
print("Examples:", run_meta.get("n_examples_evaluable"))

Loaded artifacts from /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval
Methods: ['raw', 'popularity_train', 'multi_mean_train']
Examples: 12500


In [2]:
print("## Overall leaderboard")
display(tables["overall"].sort_values(["NDCG@K", "MAP@K", "MRR"], ascending=False))

print("## Slice leaderboard")
display(tables["by_slice"].sort_values(["slice_name", "NDCG@K", "Hit@K"], ascending=[True, False, False]))

print("## Support-bucket leaderboard")
display(tables["by_support"].sort_values(["train_support_bucket", "NDCG@K"], ascending=[True, False]))

## Overall leaderboard


,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
0,popularity_train,0.15112,0.146796,0.050594,0.073109,0.073756,0.212335,0.034921,5.317471,0.000000
1,multi_mean_train,0.08328,0.076479,0.025349,0.037700,0.041694,0.146727,1.000000,9.798976,0.976841
2,raw,0.07168,0.066606,0.024917,0.035020,0.040381,0.160723,1.000000,9.838045,0.979470


## Slice leaderboard


,slice_name,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
0,slice_a_multi_target,multi_mean_train,0.188966,0.071700,0.021733,0.044159,0.075772,0.137577,0.955556,9.940777,0.982319
1,slice_a_multi_target,raw,0.143448,0.055964,0.021677,0.038812,0.073125,0.161907,0.996825,9.955937,0.983968
2,slice_a_multi_target,popularity_train,0.128276,0.053728,0.019418,0.035444,0.076750,0.212240,0.034921,5.312688,0.000000
3,slice_b_single_target,popularity_train,0.152527,0.152527,0.052514,0.075428,0.073572,0.212341,0.034921,5.317765,0.000000
4,slice_b_single_target,multi_mean_train,0.076773,0.076773,0.025572,0.037302,0.039596,0.147290,1.000000,9.790245,0.976504
5,slice_b_single_target,raw,0.067261,0.067261,0.025117,0.034786,0.038365,0.160650,1.000000,9.830787,0.979193


## Support-bucket leaderboard


,train_support_bucket,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
0,0,popularity_train,0.174720,0.174720,0.061863,0.087912,0.082621,0.212432,0.034921,5.319178,0.000000
1,0,raw,0.070720,0.070560,0.026194,0.036359,0.039677,0.163086,1.000000,9.789457,0.976578
2,0,multi_mean_train,0.070720,0.070560,0.026194,0.036359,0.039677,0.163086,1.000000,9.789457,0.976578
3,1,popularity_train,0.181120,0.181120,0.062714,0.089706,0.084369,0.212320,0.034921,5.319652,0.000000
4,1,raw,0.072640,0.072640,0.028735,0.038878,0.041850,0.159206,1.000000,9.845860,0.979483
5,1,multi_mean_train,0.080000,0.080000,0.025766,0.038187,0.039650,0.148395,1.000000,9.731983,0.974801
6,2-3,popularity_train,0.136804,0.136666,0.045143,0.066066,0.066652,0.212267,0.034921,5.316660,0.000000
7,2-3,multi_mean_train,0.080033,0.079895,0.025602,0.038089,0.040271,0.139550,0.996825,9.796552,0.975752
8,2-3,raw,0.065633,0.065494,0.022619,0.032475,0.036046,0.159345,1.000000,9.834856,0.979690
9,4-7,popularity_train,0.107238,0.086947,0.030357,0.045565,0.060413,0.212332,0.034921,5.313977,0.000000


In [3]:
print("## Popularity decile performance")
display(tables["by_pop_decile"].sort_values(["pos_pop_decile", "NDCG@K"], ascending=[True, False]))

print("## Delta vs popularity")
display(tables["pop_delta"].sort_values(["pos_pop_decile", "method"]))

print("## Personalization diagnostics")
display(tables["personalization"].sort_values("method"))

## Popularity decile performance


,pos_pop_decile,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
0,D1,multi_mean_train,0.116335,0.112709,0.036464,0.054270,0.053118,0.144198,1.000000,9.863927,0.980512
2,D1,raw,0.102789,0.101195,0.037011,0.052016,0.052139,0.159045,1.000000,9.898705,0.982913
1,D1,popularity_train,0.000000,0.000000,0.000000,0.000000,0.005287,0.212278,0.034921,5.314266,0.000000
4,D10,popularity_train,1.000000,0.997024,0.479675,0.605472,0.482485,0.212579,0.034921,5.321464,0.000000
3,D10,multi_mean_train,0.077679,0.077232,0.020990,0.033912,0.038313,0.149687,0.993651,9.721016,0.971090
5,D10,raw,0.066071,0.066071,0.019091,0.029791,0.034906,0.160321,1.000000,9.765601,0.975867
6,D2,multi_mean_train,0.091566,0.083836,0.024912,0.039153,0.041111,0.145293,0.996825,9.811335,0.979079
8,D2,raw,0.074699,0.069980,0.025202,0.036067,0.041245,0.159901,1.000000,9.833473,0.979169
7,D2,popularity_train,0.000000,0.000000,0.000000,0.000000,0.007728,0.212295,0.034921,5.315455,0.000000
9,D3,multi_mean_train,0.090698,0.079780,0.029508,0.042221,0.048729,0.145195,1.000000,9.884718,0.981515


## Delta vs popularity


,pos_pop_decile,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR,Hit@K_pop_ref,Recall@K_pop_ref,MAP@K_pop_ref,NDCG@K_pop_ref,MRR_pop_ref,Hit@K_delta_vs_pop,Recall@K_delta_vs_pop,MAP@K_delta_vs_pop,NDCG@K_delta_vs_pop,MRR_delta_vs_pop,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
0,D1,multi_mean_train,0.116335,0.112709,0.036464,0.054270,0.053118,0.000000,0.000000,0.000000,0.000000,0.005287,0.116335,0.112709,0.036464,0.054270,0.047831,0.144198,1.000000,9.863927,0.980512
1,D1,popularity_train,0.000000,0.000000,0.000000,0.000000,0.005287,0.000000,0.000000,0.000000,0.000000,0.005287,0.000000,0.000000,0.000000,0.000000,0.000000,0.212278,0.034921,5.314266,0.000000
2,D1,raw,0.102789,0.101195,0.037011,0.052016,0.052139,0.000000,0.000000,0.000000,0.000000,0.005287,0.102789,0.101195,0.037011,0.052016,0.046852,0.159045,1.000000,9.898705,0.982913
3,D10,multi_mean_train,0.077679,0.077232,0.020990,0.033912,0.038313,1.000000,0.997024,0.479675,0.605472,0.482485,-0.922321,-0.919792,-0.458685,-0.571560,-0.444172,0.149687,0.993651,9.721016,0.971090
4,D10,popularity_train,1.000000,0.997024,0.479675,0.605472,0.482485,1.000000,0.997024,0.479675,0.605472,0.482485,0.000000,0.000000,0.000000,0.000000,0.000000,0.212579,0.034921,5.321464,0.000000
5,D10,raw,0.066071,0.066071,0.019091,0.029791,0.034906,1.000000,0.997024,0.479675,0.605472,0.482485,-0.933929,-0.930952,-0.460584,-0.575681,-0.447579,0.160321,1.000000,9.765601,0.975867
6,D2,multi_mean_train,0.091566,0.083836,0.024912,0.039153,0.041111,0.000000,0.000000,0.000000,0.000000,0.007728,0.091566,0.083836,0.024912,0.039153,0.033383,0.145293,0.996825,9.811335,0.979079
7,D2,popularity_train,0.000000,0.000000,0.000000,0.000000,0.007728,0.000000,0.000000,0.000000,0.000000,0.007728,0.000000,0.000000,0.000000,0.000000,0.000000,0.212295,0.034921,5.315455,0.000000
8,D2,raw,0.074699,0.069980,0.025202,0.036067,0.041245,0.000000,0.000000,0.000000,0.000000,0.007728,0.074699,0.069980,0.025202,0.036067,0.033517,0.159901,1.000000,9.833473,0.979169
9,D3,multi_mean_train,0.090698,0.079780,0.029508,0.042221,0.048729,0.000000,0.000000,0.000000,0.000000,0.010556,0.090698,0.079780,0.029508,0.042221,0.038173,0.145195,1.000000,9.884718,0.981515


## Personalization diagnostics


,method,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
0,multi_mean_train,0.146727,1.000000,9.798976,0.976841
1,popularity_train,0.212335,0.034921,5.317471,0.000000
2,raw,0.160723,1.000000,9.838045,0.979470


In [4]:
print("## Run metadata")
run_meta

## Run metadata


{'split_requested': 'val',
 'split_used': 'val',
 'active_cohort': 'all',
 'max_examples': 12500,
 'n_examples_evaluable': 12500,
 'methods_requested': ['raw', 'popularity_train', 'multi_mean_train'],
 'methods_run': ['raw', 'popularity_train', 'multi_mean_train'],
 'k_final': 10,
 'k_personalization': 10,
 'random_seed': 2026,
 'coverage': {'n_total': 12500.0,
  'n_multi_pos': 725.0,
  'n_single_pos': 11775.0,
  'n_zero_pos': 0.0,
  'coverage_multi_pos': 0.058},
 'prep_diagnostics': {'eval_records_count': 1400227,
  'sampled_rows_count': 12500,
  'full_eval_user_count': 1151316,
  'full_eval_multi_pos_user_count': 237217,
  'sampled_rows': 12500,
  'evaluable_examples': 12500,
  'dropped_rows': 0,
  'drop_reasons': {'no_other_positive_app': 0}},
 'counts_by_slice': {'slice_a_multi_target': 725,
  'slice_b_single_target': 11775},
 'counts_by_support_bucket': {'0': 3125,
  '1': 3125,
  '2-3': 3611,
  '4-7': 2639,
  '8+': 0},
 'timing_seconds': {'prepare_inputs': 525.148,
  'score_method